# Phase 4 single-seed extension: B1 joint run (Kaggle)

Runs B1 once at seed 42. B1 jointly trains on the canonical ten-language set and is order-independent; the runner records its order as `order-independent`. The notebook passes order-1 only as the CLI routing value required by the runner.

Attach `phase4_full_baselines_code.zip`, enable Internet and a GPU, accept the gated model terms for `meta-llama/Llama-3.2-1B`, and add `HF_TOKEN` as a Kaggle secret. Attach this batch's previous output ZIP only when resuming. Each run cell saves a checkpoint archive after the method finishes; save/download Kaggle output before ending the session.

In [1]:
from pathlib import Path
import os
import subprocess
import sys
from zipfile import ZipFile

CODE_ARCHIVE_NAME = "phase4_full_baselines_code.zip"
INPUT_ROOT = Path("/kaggle/input")
source_files = list(INPUT_ROOT.rglob("phase4/run_phase4.py"))
source_roots = {path.parents[1] for path in source_files}
if len(source_roots) == 1:
    REPO_ROOT = source_roots.pop()
elif source_roots:
    raise RuntimeError(f"Found multiple Phase 4 source trees: {sorted(map(str, source_roots))}")
else:
    code_archives = list(INPUT_ROOT.rglob(CODE_ARCHIVE_NAME))
    if len(code_archives) != 1:
        raise FileNotFoundError(f"Attach exactly one {CODE_ARCHIVE_NAME} Kaggle input; found: {code_archives}")
    extraction_root = Path("/kaggle/working/phase4_full_baselines_source")
    with ZipFile(code_archives[0]) as archive:
        archive.extractall(extraction_root)
    source_files = list(extraction_root.rglob("phase4/run_phase4.py"))
    if len(source_files) != 1:
        raise FileNotFoundError(f"Expected one bundled phase4/run_phase4.py; found: {source_files}")
    REPO_ROOT = source_files[0].parents[1]
PROTOCOL_RELATIVE_PATH = Path("phase4/configs/controlled_10lang_full_methods_seed42_extension_protocol.json")
if not (REPO_ROOT / PROTOCOL_RELATIVE_PATH).is_file():
    raise FileNotFoundError("The attached code bundle does not contain the approved extension protocol.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "phase2" / "requirements-phase2.txt")], check=True)
sys.path.insert(0, str(REPO_ROOT))

from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN Kaggle secret is empty.")
os.environ["HF_TOKEN"] = hf_token
del hf_token
print({"source": str(REPO_ROOT), "code_archive": CODE_ARCHIVE_NAME})


{'source': '/kaggle/input/datasets/seethaganesh/phase4-full-baselines-code', 'code_archive': 'phase4_full_baselines_code.zip'}


In [2]:
import json
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

from phase2.protocol import build_manifest, load_protocol
from phase2.results import aggregate_results, validate_result_record, write_result_record
from phase2.run_baseline import validate_prepared_artifacts
from phase4.run_phase4 import _ensure_prepared_data, run_selected_pair

PROTOCOL_PATH = REPO_ROOT / PROTOCOL_RELATIVE_PATH
INPUT_ROOT = Path("/kaggle/input")
TRAINING_SEED = 42
ALL_METHODS = ["B0", "B1", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "M3"]
EXTENSION_METHODS = ["B0", "B1", "B3", "B4", "B6", "B7", "B8"]

BATCH_ID = "b1_joint"
METHOD_IDS = ["B1"]
ORDER_ID = "order-1"
ORDER_LABEL = "order-independent"

protocol = load_protocol(PROTOCOL_PATH)
gate = protocol["phase4"]["prior_gate"]
if protocol["run_methods"] != EXTENSION_METHODS:
    raise ValueError("The extension protocol run_methods list does not match this notebook set.")
if gate["status"] != "passed" or gate["approved_methods"] != ALL_METHODS:
    raise RuntimeError("The supervisor-approved full-method extension gate is missing.")
if protocol["phase4"]["comparison"]["methods"] != ALL_METHODS:
    raise ValueError("The comparison roster differs from the approved ten methods.")
if protocol["training"]["seeds"] != [TRAINING_SEED]:
    raise ValueError("This notebook set is frozen for seed 42 only.")
if not set(METHOD_IDS) <= set(EXTENSION_METHODS):
    raise ValueError("This notebook contains a method outside the extension run set.")
if ORDER_LABEL != "order-independent" and ORDER_ID not in {order["id"] for order in protocol["orders"]}:
    raise ValueError(f"Unknown order ID: {ORDER_ID}")

ARTIFACT_ROOT = Path("/kaggle/working") / f"phase4_full_methods_seed42_{BATCH_ID}"
RESULTS_DIR = ARTIFACT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
_ensure_prepared_data(PROTOCOL_PATH, ARTIFACT_ROOT)
prepared = validate_prepared_artifacts(protocol, ARTIFACT_ROOT)
manifest = build_manifest(protocol)
manifest["data_artifacts"] = prepared["data_artifacts"]
manifest["data_manifest_sha256"] = prepared["data_manifest_sha256"]

CHECKPOINT_NAME = f"phase4_full_methods_seed42_{BATCH_ID}.zip"
OUTPUT_ARCHIVE = Path("/kaggle/working") / CHECKPOINT_NAME

def restore_checkpoint_archives():
    for checkpoint_path in sorted(INPUT_ROOT.rglob(CHECKPOINT_NAME)):
        with ZipFile(checkpoint_path) as archive:
            if "protocol_manifest.json" not in archive.namelist():
                raise ValueError(f"Checkpoint lacks protocol_manifest.json: {checkpoint_path}")
            saved_manifest = json.loads(archive.read("protocol_manifest.json"))
            if (
                saved_manifest.get("protocol_id") != manifest["protocol_id"]
                or saved_manifest.get("protocol_sha256") != manifest["protocol_sha256"]
                or saved_manifest.get("data_manifest_sha256") != prepared["data_manifest_sha256"]
            ):
                raise ValueError(f"Checkpoint protocol/data hash mismatch: {checkpoint_path}")
            for member in archive.namelist():
                if not member.startswith("results/") or not member.endswith(".json"):
                    continue
                record = json.loads(archive.read(member))
                validate_result_record(record, manifest)
                if record["method_id"] not in METHOD_IDS:
                    raise ValueError(f"Checkpoint contains a result outside this notebook: {record['run_id']}")
                destination = RESULTS_DIR / f"{record['run_id']}.json"
                if destination.exists():
                    existing = json.loads(destination.read_text(encoding="utf-8"))
                    if existing != record:
                        raise ValueError(f"Conflicting saved result: {record['run_id']}")
                else:
                    write_result_record(record, RESULTS_DIR, manifest)


def save_checkpoint_archive():
    report = aggregate_results(manifest, RESULTS_DIR)
    result_paths = sorted(RESULTS_DIR.glob("*.json"))
    with ZipFile(OUTPUT_ARCHIVE, "w", ZIP_DEFLATED) as archive:
        archive.write(ARTIFACT_ROOT / "protocol_manifest.json", "protocol_manifest.json")
        for result_path in result_paths:
            record = json.loads(result_path.read_text(encoding="utf-8"))
            validate_result_record(record, manifest)
            if record["method_id"] not in METHOD_IDS:
                raise ValueError(f"Unexpected method in batch output: {record['method_id']}")
            archive.write(result_path, f"results/{result_path.name}")
        archive.writestr("aggregate.json", json.dumps(report, indent=2, sort_keys=True) + "\n")
    print(f"Checkpoint saved: {OUTPUT_ARCHIVE}; {len(result_paths)} run record(s) in this notebook.")

restore_checkpoint_archives()
print({
    "batch_id": BATCH_ID,
    "methods": METHOD_IDS,
    "order": ORDER_LABEL,
    "training_seed": TRAINING_SEED,
    "extension_run_count": manifest["expected_run_count"],
    "data_manifest_sha256": prepared["data_manifest_sha256"],
    "restored_records": len(list(RESULTS_DIR.glob("*.json"))),
})

def run_method_and_checkpoint(method_id):
    if method_id not in METHOD_IDS:
        raise ValueError(f"Method {method_id!r} is not scheduled in this notebook.")
    result = run_selected_pair(
        method_id=method_id,
        order_id=ORDER_ID,
        training_seed=TRAINING_SEED,
        protocol_path=PROTOCOL_PATH,
        artifact_root=ARTIFACT_ROOT,
    )
    method_records = result if isinstance(result, list) else [result]
    for record in method_records:
        validate_result_record(record, manifest)
    save_checkpoint_archive()
    return [
        {"run_id": record["run_id"], "ACC": record["metrics"]["ACC"]}
        for record in method_records
    ]


Preparing the frozen XNLI subsets...


README.md:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

en/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 50.2MB            

en/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

en/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  308kB            

en/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (83 outputs left out)

Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5010 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2490 [00:00<?, ? examples/s]

{'batch_id': 'b1_joint', 'methods': ['B1'], 'order': 'order-independent', 'training_seed': 42, 'extension_run_count': 31, 'data_manifest_sha256': '373852a7f1e09ab99da8a25f21023fedc1977ab1969e09d938874f7c645a721a', 'restored_records': 0}


In [3]:
run_method_and_checkpoint("B1")


Starting xnli-phase4-full-methods-10lang-500x200-seed42-extension-v1-B1-order-independent-seed-42


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

... (5 outputs left out)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Step,Training Loss
10,1.559056
20,1.092978
30,1.037982
40,0.952669
50,1.029154
60,0.809801
70,0.841114
80,0.896672
90,0.758812
100,0.772699


Completed xnli-phase4-full-methods-10lang-500x200-seed42-extension-v1-B1-order-independent-seed-42: ACC=0.6930
Checkpoint saved: /kaggle/working/phase4_full_methods_seed42_b1_joint.zip; 1 run record(s) in this notebook.


[{'run_id': 'xnli-phase4-full-methods-10lang-500x200-seed42-extension-v1-B1-order-independent-seed-42',
  'ACC': 0.693}]

In [4]:
completed_records = [
    json.loads(path.read_text(encoding="utf-8"))
    for path in RESULTS_DIR.glob("*.json")
]
expected_batch_runs = [
    run for run in manifest["runs"]
    if run["method_id"] in METHOD_IDS
    and (run["method_id"] == "B1" or run["order_id"] == ORDER_ID)
]
expected_batch_ids = {run["run_id"] for run in expected_batch_runs}
completed_batch_ids = {record["run_id"] for record in completed_records}
missing_batch_ids = sorted(expected_batch_ids - completed_batch_ids)
if missing_batch_ids:
    raise RuntimeError(f"This notebook batch is incomplete: {missing_batch_ids}")
print({
    "batch_id": BATCH_ID,
    "completed_this_notebook": len(completed_batch_ids),
    "expected_this_notebook": len(expected_batch_ids),
    "checkpoint_archive": str(OUTPUT_ARCHIVE),
})


{'batch_id': 'b1_joint', 'completed_this_notebook': 1, 'expected_this_notebook': 1, 'checkpoint_archive': '/kaggle/working/phase4_full_methods_seed42_b1_joint.zip'}
